In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'SimHei'  # 设置中文字体
plt.rcParams['axes.unicode_minus'] = False  # 确保负号字符可用

In [ ]:
x = np.linspace(0, 10, 20) + np.random.randn(20)
y = np.linspace(0, 10, 20) + np.random.randn(20)
plt.scatter(x, y)

In [ ]:
X = np.hstack((np.ones((20, 1)), x.reshape(-1, 1)))
X

In [ ]:
theta = np.dot(np.dot(np.linalg.inv(np.dot(X.T, X)), X.T), y)
theta

In [ ]:
plt.scatter(x, y)
x_test = np.linspace(-2, 10, 10)
y_test = theta[1] * x_test + theta[0]
plt.plot(x_test, y_test, c='r')

sklearn的线型回归

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
linear = LinearRegression()
linear.fit(x.reshape(-1, 1), y)

In [ ]:
linear.coef_

In [ ]:
linear.intercept_

# 梯度下降

批量梯度下降

随机梯度下降

小批量梯度下降

In [ ]:
# 定义一个方法，将画布设置为数学样式。
def math_style(ax):
    # 隐藏上侧，右侧默认边框
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # 左侧，下侧坐标轴移动到（0，0）
    ax.spines['left'].set_position(('data', 0))
    ax.spines['bottom'].set_position(('data', 0))
    # 加上坐标轴的箭头
    ax.plot(1, 0, ">", transform=ax.get_yaxis_transform(), clip_on=False, markersize=6)
    ax.plot(0, 1, "^", transform=ax.get_xaxis_transform(), clip_on=False, markersize=6)
    # 设置equal等比例坐标的同时，不修改坐标范围，只拉伸画布。
#     ax.set_aspect('equal', adjustable='box') 
    # 网格
    ax.grid(True, alpha=0.3, linestyle='--')
    # x,y轴标签
    ax.set_xlabel('x', loc='right')
    ax.set_ylabel('y', loc='top', rotation=0)

In [ ]:
# 可以看到f(x)在2.5处达到最小值。导函数也在2.5处值为0
x = np.linspace(-10, 10, 1000)
f = lambda x : (x-2.5) ** 2 + 5
y = f(x)
fig, ax = plt.subplots()
ax.plot(x, y)
ax.plot(x, 2 * x - 5, c='r')
math_style(ax)

下面通过梯度下降来求f(x)的最小值位置的x是多少？看是否和求导获得的结果一致。

这个案例没什么必要使用梯度下降求最小值位置，  
但很多案例里的函数是高维的，甚至有多个极值，这时就可以用到梯度下降了。

### 梯度下降：沿着函数下降最快的方向，一步步挪动参数x，直到f(x)达到最小值。

梯度下降的计算方法：x - rate * dx(x)

1. 为什么要用导数来移动：  
导数指向函数变化方向和变化量。  
导数为负，表示函数向右减小，极小值在右侧。
导数为正，表示函数向左减小，极小值在左侧。
越靠近极小值，导数值越小，移动越慢，趋近于0，容易接近极小值。

2. 为什么要用减号：  
导数负，减号表示向右。  
导数正，减号表示向左。  
是极小值的方向

3. 为什么要乘以学习率:  
    导数的值可能会很大，为了防止直接跨过极值点，需要控制步长，乘以一个较小系数可以解决。

In [ ]:
dx = lambda x: 2 * x -5

In [ ]:
# tidu函数是用来对导函数进行梯度下降运算的。
# dx：原函数的导函数
# x_ori：x的起始点，一般随便取一个点即可
# rate：梯度下降的速率
# max_iter：最大迭代次数
# threshold：梯度下降的最小阈值。
# quit_mode：退出模式（1：迭代次数；2：梯度阈值）
# ret：每次梯度下降的结果存起来返回。
def tidu(dx, x_ori, rate=0.01, max_iter=2000, threshold=1e-5, quit_mode=1, ret = None):
    if ret is None:
        ret = []
    # 迭代次数计数
    i = 0
    while True:
        # 保存每次结果
        ret.append(x_ori)
        # 计算一次下降的结果
        x_ori = x_ori - rate * dx(x_ori)
        # 更新迭代次数
        i += 1
        #退出条件
        # 1. 达到最大迭代次数退出
        if quit_mode == 1 and i > max_iter:
            break
        # 2. 梯度小于最小阈值退出
        if quit_mode == 2 and abs(dx(x_ori)) < threshold:
            break
    return ret

In [ ]:
# 达到最大迭代次数退出
ret = tidu(dx, -10)
ret

In [ ]:
len(ret)

In [ ]:
ret[:5]

In [ ]:
ret[-1]

In [ ]:
# 最小阈值退出
ret = tidu(dx, -10, quit_mode=2)
ret

In [ ]:
len(ret)

In [ ]:
ret[:5]

In [ ]:
ret[-1]

In [ ]:
# 学习率调高
ret = tidu(dx, -10, rate=0.05, quit_mode=2)
ret

In [ ]:
display(len(ret), ret[:5], ret[-1])

In [ ]:
x = np.linspace(-10, 10, 1000)
f = lambda x : (x-2.5) ** 2 + 5
y = f(x)
fig, ax = plt.subplots()
ax.plot(x, y)
ax.scatter(ret,[f(t) for t in ret], c='r')

math_style(ax)

In [ ]:
# 学习率再调高，会出现执行停止不了的情况。手动中断内核运行
# 这里因为等不到方法返回，所以把数组对象作为参数传入，这样方法执行被打断也可以获取ret值。
ret = []
tidu(dx, -10, rate=1, quit_mode=2, ret=ret)

In [ ]:
# 从ret的长度可以看出循环了非常多次。
# 从前五个ret值可以看出x一直在-10和15之间摇摆。
# 说明学习率过高会导致每次梯度下降过头，出现无法接近极值处的情况。
display(len(ret), ret[:5], ret[-1])

从学习率为1的场景可以看出，对于x^2这种堆成的函数。x如果一次下降dx的话，会跑到对称位置。

In [ ]:
# 学习率再调高呢？
ret = []
tidu(dx, -10, rate=1.1, quit_mode=2, ret=ret)

In [ ]:
# 可以看出x的绝对值一直在变大，每次都离极值点更远了。
display(len(ret), ret[:5], ret[-1])

In [ ]:
# 数据量太大，取前20个点观察即可
ret2 = ret[:25]
ret2

In [ ]:
# 画图看看
x = np.linspace(-1000, 1000, 1000)
f = lambda x : (x-2.5) ** 2 + 5
y = f(x)
fig, ax = plt.subplots()
ax.plot(x, y)
ax.plot(ret2,[f(t) for t in ret2], c='r')

math_style(ax)

可以看出x在左右摇摆中很快增大，越来越远离极小点。

In [ ]:
# 如果学习率调的比1低一点呢。
ret = tidu(dx, -10, rate=0.9, quit_mode=2)

In [ ]:
display(len(ret), ret[:5], ret[-1])

In [ ]:
x = np.linspace(-10, 15, 1000)
f = lambda x : (x-2.5) ** 2 + 5
y = f(x)
fig, ax = plt.subplots()
ax.plot(x, y)
ax.plot(ret,[f(t) for t in ret], c='r')

math_style(ax)

从图中可以看出，虽然每次x会跨过极值点，但每次会更接近极值点一些，最终还是会逼近极值点。

### 线性回归实例：糖尿病

In [ ]:
# 导数据包
from sklearn.datasets import load_diabetes

In [ ]:
# 获取数据
diabetes = load_diabetes()
data = diabetes.data
target = diabetes.target
feature_names = diabetes.feature_names
display(data, target, feature_names)

In [ ]:
data.shape

In [ ]:
# 目标值非常分散，适合回归而非分类。
np.unique(target)

In [ ]:
df = pd.DataFrame(data=data, columns=feature_names)
df

In [ ]:
print(diabetes.DESCR)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(data, target)
display(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
# 训练
linear = LinearRegression()
linear.fit(X_train, y_train)

In [ ]:
# 预测
y_ = linear.predict(X_test)

In [ ]:
# 评估
linear.score(X_test, y_test)

$ R^2 = 1 - \frac{\sum (y_i - \hat{y_i})^2}{\sum (y_i - \bar{y})^2} $

$( y_i ) 是实际观测值；$

$( \hat{y_i} ) 是模型预测值；$

$( \bar{y} ) 是实际观测值的平均值。$

In [ ]:
# 手动评分
1 - (((y_test - y_) ** 2).sum() / ((y_test - y_test.mean()) ** 2).sum())

In [ ]:
# 回归的线性方程系数
linear.coef_

In [ ]:
linear.intercept_

$
y = \beta_0 + \beta_1 \cdot x_1 + \beta_2 \cdot x_2 + \ldots + \beta_n \cdot x_n
$

In [ ]:
# 线型回归得到线型方程。
f = lambda x : linear.intercept_ + (linear.coef_ * x).sum()

In [ ]:
# 从测试集里取一条数据和对应的预测结果。
display(X_test[0], y_[0])

In [ ]:
# 用上面的方程手动预测一个值看看。
f(X_test[0])

画出每个特征点分布和线型方程的关系。

In [ ]:
# 提取单个特征的系数向量
# np.arrange：创建一个1到len(coef)的数组
# == i ：当前i位置为true，其他位置为false的数组。
# coef * ：在做运算时，true被视为1，false被视为0.
# 最终得到第i位置的值不变，其他位置为0的向量。
coef = lambda i: linear.coef_ * (np.arange(1, len(linear.coef_)+1) == i)

In [ ]:
# 创建5行两列的子画布，每个画布长6，宽4
fig, axs = plt.subplots(5, 2, figsize=(6 * 2, 4 *5))
axs = axs.flatten()
# 遍历所有特征列
for i, col in enumerate(feature_names, 1):
    # 当前特征的值
    X_train[:, i-1]
    # 当前特征的系数向量
    coef(i)
    # 当前特征对应的方程
    fs = lambda x : np.dot(x, coef(i)) + linear.intercept_
    # 绘图
    axs[i-1].plot(X_train[:, i-1], np.array([fs(X_train[row]) for row in np.arange(len(X_train))]))
    axs[i-1].set_title(col)
    axs[i-1].set_ylim((target.min(), target.max()))

### 提取单个特征系数的方程来绘图不是个好方法

### 回归方程在某个特征上的系数不为0不代表该特征和结果有相关性。

下面的regplot相当于，单独对每个特征做一次线性回归，画出数据散点图和回归方程线。

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt



# 画 特征x 与 严重程度y 的真实关系
for i, col in enumerate(feature_names, 1):
    sns.regplot(x=df[col].values, y=target, 
                scatter_kws={'alpha':0.3}, 
                line_kws={'color':'red'}, ci=None)
    plt.title(f"真实特征相关性：{col}")
    plt.show()

手动对每个特征做线性回归

In [ ]:
fig, axes = plt.subplots(5, 2, figsize=(6 * 2, 4 * 5))
axes = axes.flatten()
for i, col in enumerate(feature_names, 1):
    # 线性回归
    linear = LinearRegression()
    linear.fit(df[[col]].values, target)
    
    # 线性回归图
    feat_max = df[[col]].values.max()
    feat_min = df[[col]].values.min()
    x = np.linspace(feat_min, feat_max, 1000).reshape(-1, 1)
    y = linear.predict(x)
    axes[i-1].plot(x, y, c='r')
    # 约束y轴的范围，否则看不出x变动对y的大小影响
    axes[i-1].set_ylim((target.min(), target.max()))
    
    # 特征和结果的散点图
    axes[i-1].scatter(df[col].values, target)

## 作业一：波士顿房价

In [ ]:
from sklearn.datasets import load_boston

In [ ]:
boston = load_boston()
boston

In [ ]:
print(boston.DESCR)

波士顿房价数据集
---------------------------

**数据集特征：**

    :实例数量: 506 

    :属性数量: 13个数值/类别预测属性。中位值（属性14）通常是目标属性。

    :属性信息（按顺序）:
        - CRIM     每个城镇的人均犯罪率
        - ZN       每个城镇占地超过25,000平方英尺的住宅用地比例
        - INDUS    每个城镇非零售业务用地比例
        - CHAS     查尔斯河虚拟变量（如果区域邻近河流则为1，否则为0）
        - NOX      一氧化氮浓度（百万分之几）
        - RM       每个住所的房间平均数
        - AGE      1940年前建造的业主自住单位比例
        - DIS      到波士顿五个就业中心的加权距离
        - RAD      径向高速公路的可及性指数
        - TAX      每10,000美元的全值房产税率
        - PTRATIO  每个城镇的师生比例
        - B        其中Bk是每个城镇黑人比例，B = 1000*(Bk - 0.63)^2
        - LSTAT    人口中较低社会地位的比例
        - MEDV     业主自住房屋的中位价值（以千美元计）

    :缺失属性值: 无

    :创建者: Harrison, D. 和 Rubinfeld, D.L.

这是UCI ML房屋数据集的副本。
https://archive.ics.uci.edu/ml/machine-learning-databases/housing/

该数据集来自卡内基梅隆大学维护的StatLib库。

Harrison, D. 和 Rubinfeld, D.L. 的波士顿房价数据出自 'Hedonic
prices and the demand for clean air'，J. Environ. Economics & Management,
vol.5, 81-102, 1978。   该数据集被用于 Belsley, Kuh & Welsch, 'Regression diagnostics
...', Wiley, 1980。   注意：后者一书的244-261页的表格中使用了各种转换。

波士顿房价数据已被用于许多解决回归问题的机器学习论文中。
     
.. topic:: 参考资料

   - Belsley, Kuh & Welsch, 'Regression diagnostics: Identifying Influential Data and Sources of Collinearity', Wiley, 1980. 244-261.
   - Quinlan,R. (1993). Combining Instance-Based and Model-Based Learning. In Proceedings on the Tenth International Conference of Machine Learning, 236-243, University of Massachusetts, Amherst. Morgan Kaufmann.

In [ ]:
# 获取数据
data = boston.data
target = boston.target
feature_names = boston.feature_names

In [ ]:
df = pd.DataFrame(data, columns=feature_names)
df

In [ ]:
# 回归问题
np.unique(target)

In [ ]:
# 数据拆分
X_train, X_test, y_train, y_test = train_test_split(data, target)

In [ ]:
# 训练
linear = LinearRegression()
linear.fit(X_train, y_train)

In [ ]:
# 预测
y_ = linear.predict(X_test)

In [ ]:
# 评分
linear.score(X_test, y_test)

In [ ]:
# 线性回归方程
f = lambda x : np.dot(x, linear.coef_) + linear.intercept_

In [ ]:
# 选一个数据手动预测
display(X_test[0], y_test[0], y_[0])

In [ ]:
f(X_test[0])

In [ ]:
# 手动评分
1 - (((y_test - y_) ** 2).sum()) / (((y_test - y_test.mean()) ** 2).sum())

In [ ]:
# 单个特征进行线性回归并绘图
fig, axes = plt.subplots(7, 2, figsize=(6 * 2, 4 * 7))
axes = axes.flatten()
# 多个一个子图，设置最后一个子图不显示
axes[-1].set_visible(False)
for i, col in enumerate(feature_names):
    # 单个特征训练
    linear = LinearRegression()
    linear.fit(df[[col]], target)
    
    # 线性方程图
    x = np.linspace(df[col].min(), df[col].max(), 1000).reshape(-1, 1)
    y = linear.predict(x)
    axes[i].plot(x, linear.predict(x), c='r')
    
    # 散点图
    axes[i].scatter(df[col].values, target)
    
    axes[i].set_title(col)

可以看出RM（平均房间数）和LSTAT（低社会地位人口比例）和房价相关性比较高

至于B（其中Bk是每个城镇黑人比例，B = 1000*(Bk - 0.63)^2）

这个特征，因为是差值的平方，可以看出黑人比例和0.63的差值越大，房价变高和变低的概率越高。

如果这个特征不是差值平方，而是Bk，可能更能看出黑人比例和房价的相关性。

## 作业二：预测鲍鱼年龄 abalone.txt

In [ ]:
df = pd.read_csv('../day25_线性回归(四)/代码/data/abalone.txt', sep='\t', header=None)
df

可以看出，前8列是特征，最后一列是目标值。

第一行不是特征名，因此特征的含义未知。

In [ ]:
data = df.iloc[:, :-1].values
target = df.iloc[:, -1].values

In [ ]:
# 回归问题
np.unique(target)

In [ ]:
# 拆分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(data, target)

In [ ]:
# 训练
linear = LinearRegression()
linear.fit(X_train, y_train)

In [ ]:
# 预测
y_ = linear.predict(X_test)

In [ ]:
# 评分
linear.score(X_test, y_test)

In [ ]:
# 手动预测
f = lambda x : np.dot(x, linear.coef_) + linear.intercept_

In [ ]:
display(X_test[0], y_test[0], y_[0], f(X_test[0]))

In [ ]:
# 手动评分
1 - (((y_test - y_) ** 2).sum()) / (((y_test - y_test.mean()) ** 2).sum())

In [ ]:
# 对单个特征进行回归并画图
fig, axes = plt.subplots(4, 2, figsize=(6 * 2, 4 * 4))
axes = axes.flatten()
for i in range(8):
    # 单特征训练
    linear = LinearRegression()
    linear.fit(df[[i]], target)
    
    # 线型回归图
    x = np.linspace(df[i].min(), df[i].max(), 1000).reshape(-1, 1)
    y = linear.predict(x)
    axes[i].plot(x, y, c='r')
    
    # 散点图
    axes[i].scatter(df[i].values, target)

可以看出，除了第一个特征，其他特征都和目标值由比较明显的相关性。

## 作业三：Real estate valuation dataset.xlsx

这个数据集文件的中文是房地产估价数据集。

In [ ]:
df = pd.read_excel('../day25_线性回归(四)/代码/data/Real estate valuation data set.xlsx', engine='openpyxl')
df

从数据可以看出，有6个特征，第1列为数据编号，最后一列为目标值。

In [ ]:
data = df.iloc[:, 1:-1].values
target = df.iloc[:, -1].values

In [ ]:
# 拆分训练集和测试集。
# 数据量比较小，测试集少分点。
X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=40)

In [ ]:
# 训练
linear = LinearRegression()
linear.fit(X_train, y_train)

In [ ]:
# 预测
y_ = linear.predict(X_test)

In [ ]:
# 评分
linear.score(X_test, y_test)

In [ ]:
# 手动预测
f = lambda x : np.dot(linear.coef_, x) + linear.intercept_

In [ ]:
display(X_test[0], y_test[0], y_[0], f(X_test[0]))

In [ ]:
# 手动评分
1 - (((y_test - y_) ** 2).sum()) / (((y_test - y_test.mean()) ** 2).sum())

In [ ]:
# 单个特征进行回归，并绘图
fig, axes = plt.subplots(3, 2, figsize=(6 * 2, 4 * 3))
axes = axes.flatten()
for i, col in enumerate(df.iloc[:, 1:-1], 1):
    # 回归
    linear = LinearRegression()
    linear.fit(df[[col]].values, target)
    
    # 线性回归图
    x = np.linspace(df[col].min(), df[col].max(), 1000).reshape(-1, 1)
    y = linear.predict(x)
    axes[i-1].plot(x, y, c='r')
    
    # 散点图
    axes[i-1].scatter(df[col].values, target)
    
    # 标题
    axes[i-1].set_title(col)

可以看出X3,X5,X6特征和目标值相关性比较明显。

## 作业四、工业蒸汽预测（机器学习-线性回归）

# 火力发电的核心是：锅炉的燃烧效率。

探究各个影响因素对锅炉燃烧效率的影响。

阿里云，天池平台：工业蒸汽预测

In [ ]:
from scipy import stats

箱型图查看数据分布

In [ ]:
data = np.random.randn(100)
print(data)
plt.boxplot(data)
plt.show()

kde图查看训练集和测试集分布差异

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

data = np.random.randn(100)
sns.kdeplot(data)
plt.show()


删除训练集和测试集分布明显差异比较大的特征。

使用seabon绘制相关性热力图

左下角相关性热力图

删除相关性不明显的特征数据